# `aidp-alh` live test — wallet (mTLS)

**Live-test row 1.** Reads a known ALH table via Spark JDBC using a wallet.

**Prerequisites:** `ALH_*` env vars set or OCI Vault configured. ALH wallet ZIP available.


In [ ]:
import sys, os
# Adjust this if you've uploaded the plugin scripts/ dir elsewhere.
sys.path.insert(0, '/Workspace/Shared/oracle_ai_data_platform_connectors/scripts')


In [ ]:
from oracle_ai_data_platform_connectors.auth import write_wallet_to_tmp
from oracle_ai_data_platform_connectors.jdbc import build_oracle_jdbc_url, spark_jdbc_options_wallet

tns_admin = write_wallet_to_tmp(
    wallet=os.environ.get('ALH_WALLET_ZIP_PATH', '/tmp/alh-wallet.zip'),
    target_dir='/tmp/wallet/alh',
)
url = build_oracle_jdbc_url(tns_alias=os.environ['ALH_TNS_SERVICE'], tns_admin=tns_admin)
opts = spark_jdbc_options_wallet(url=url, user=os.environ['ALH_USER'], password=os.environ['ALH_PASSWORD'])


In [ ]:
df = spark.read.format('jdbc').options(**opts).option('dbtable', os.environ['ALH_TABLE_FOR_TEST']).load()
df.show(5)


In [ ]:
# Live-test driver parses this final cell's stdout for the JSON summary.
import json, time
summary = {
    'connector': 'aidp-alh',
    'auth': 'wallet',
    'rows': int(df.count()),
    'schema': sorted([f.name for f in df.schema.fields]),
    'timestamp_utc': int(time.time()),
}
print('AIDP_LIVE_TEST_RESULT_BEGIN')
print(json.dumps(summary, indent=2))
print('AIDP_LIVE_TEST_RESULT_END')
